###  CREATE DATAFRAME

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import *

In [0]:
data = [(1,	1)
, (2,	1)
, (3,	1)
, (4,	2)
, (5,	1)
, (6,	2)
, (7,	2)]

In [0]:
schema= ['id', 'num']

In [0]:
df = spark.createDataFrame(data, schema)

In [0]:
df.display()

- What is schema
- What is dataframe
- How to select columns
- How many ways to select columns
- What is expression

Fundatamental Dataframe Operations --> Aggregations --> Join --> Window

Schema includes info about Column Name and Column Type

In [0]:
data_path = '/Workspace/Users/rashisinghvi309@gmail.com/PySparkTutorial/PysparkTutorialSeries_ByManishKumar/data/'

In [0]:
employee_df = spark.read.format('csv')\
                .option('header', 'true')\
                .option('inferSchema', 'true')\
                .option('mode', 'PERMISSIVE')\
                .load(data_path + 'employee_data.csv')
employee_df.show()

View Schema and Columns

In [0]:
employee_df.printSchema()

In [0]:
employee_df.columns

In [0]:
#DEFINE SCHEMA
emp_schema = StructType(
                [
                    StructField("id", IntegerType(), True),
                    StructField("name", StringType(), True),
                    StructField("salary", IntegerType(), True),
                    StructField("age", IntegerType(), True),
                    StructField("address", StringType(), True),
                    StructField("gender", StringType(), True),
                    StructField("_corrupt_record", StringType(), True)
                ]
)

In [0]:
employee_df_age = employee_df.select((col("age")+5).alias("AgeIncrementBy5")).show()

Select Multiple Columns

In [0]:
employee_df.select("name", "age", "salary").show()

In [0]:
employee_df.select(col("name"), col("age"), col("salary")).show()

Select column in different approaches

In [0]:
employee_df.select('id', col('name'), employee_df['salary'], employee_df.age).show()

Expressions in pyspark

In [0]:
employee_df.select(expr('id as employee_id'), expr('name as employee_name'),\
                    expr("concat(name, '_', address) as Name_Address")).show()

Change above code to Spark SQL

In [0]:
employee_df.createOrReplaceTempView('employee_tbl')

In [0]:
spark.sql("""
        SELECT id as EmployeeId, name as EmployeeName, concat(name, '_', address) as Name_Address FROM employee_tbl
""").show()

#### TRANSFORMATIONS 
- ALIASING
- FILTER / WHERE
- LITERAL
- ADDING COLUMNS
- RENAMING COLUMNS
- CASTING DATATYPES
- REMOVING COLUMNS

In [0]:
employee_df.select(col("id").alias("EmployeeId")).show()

In [0]:
employee_df.filter(col('salary') > 150000).show()

In [0]:
employee_df.where(col('salary') > 150000).show()

In [0]:
employee_df.where((col('salary') > 150000) & (col('age') <  18)).show()

In [0]:
employee_df.select("*", lit("Jain").alias("last_name")).show()

In [0]:
#Add Columns
employee_df.withColumn("lastname", lit("Singhvi")).show()

In [0]:
employee_df.withColumnRenamed('id', 'employee_id').show()

In [0]:
employee_df.withColumn('id', col('id').cast('string'))\
            .withColumn('salary', col('salary').cast('long'))\
.printSchema()

In [0]:
employee_df.drop("id", col('name')).show()

#### Convert the code into Spark SQL

In [0]:
spark.sql("""
    SELECT id as employee_id,name, cast(salary as DOUBLE) as emp_salary
    , age, address, 'Singhvi' as last_name, concat(name,' ', last_name) as full_name 
    FROM employee_tbl where age < 18 and salary > 150000
""").show()

#### DIFFERENCE BETWEEN UNION AND UNION ALL
- WHAT HAPPEN IF NUMBER OF COLUMNS CHANGED WHILE UNION OPERATION
- WHAT HAPPEN WHEN COLUMN NAME IS DIFFERENT
- WHAT IS UnionByName

In [0]:
data=[(10 ,'Anil',50000, 18),
(11 ,'Vikas',75000,  16),
(12 ,'Nisha',40000,  18),
(13 ,'Nidhi',60000,  17),
(14 ,'Priya',80000,  18),
(15 ,'Mohit',45000,  18),
(16 ,'Rajesh',90000, 10),
(17 ,'Raman',55000, 16),
(18 ,'Sam',65000,   17)]

schema = ['id', 'name', 'salary', 'manager_id']
manager_df = spark.createDataFrame(data, schema)

In [0]:
manager_df.show()

In [0]:
data1=[(19 ,'Sohan',50000, 18),
(20 ,'Sima',75000,  17)]
manager_df1 = spark.createDataFrame(data1, schema)

In [0]:
# union and unionall are equivalent in dataframe
print("Union Count:", manager_df.union(manager_df1).count())
print("Union ALL Count:", manager_df.unionAll(manager_df1).count())

In [0]:
wrong_column_data=[(19 ,50000, 18,'Sohan'),
(20 ,75000,  17,'Sima')]
wrong_column_schema = ['id', 'salary', 'manager_id', 'name']
wrong_column_df = spark.createDataFrame(wrong_column_data, wrong_column_schema)
wrong_column_df.show()

In [0]:
# manager_df.union(wrong_column_df).show()

In [0]:
manager_df.unionByName(wrong_column_df).show()

UNION OPERATION WITH ADDTIONAL COLUMNS

In [0]:
wrong_column_data1=[(19 ,50000, 18,'Sohan',10),
(20 ,75000,  17,'Sima',20)]
wrong_column_schema1 = ['id', 'salary', 'manager_id', 'name','bonus']
wrong_column_df = spark.createDataFrame(wrong_column_data1, wrong_column_schema1)
wrong_column_df.show()

In [0]:
wrong_column_df.select('id', 'name', 'manager_id', 'salary').unionByName(manager_df).show()